In [1]:
!pip install --upgrade transformers==4.45.0 huggingface_hub

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.9 MB 1.7 MB/s eta 0:00:06
   ---- ----------------------------------- 1.0/9.9 MB 1.4 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/9.9 MB 1.5 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/9.9 MB 1.3 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.9 MB 1.3 MB/s eta 0:00:07
   -------- ------------------------------- 2.1/9.9 MB 1.3 MB/s eta 0:00:07
   -------- ------------------------------- 2.1/9.9 MB 1.3 MB/s eta 0:00:07
   --------- ------------------------------ 2.4/9.9 MB 1.2 MB/s eta 0:00:07
   --------- ------------------------------ 2.4/9.9 MB 1.2 MB/s eta 0:00:07
   ---------- ----------------------------- 2.6/9.9 MB 1.1 MB/s eta 0:00:07
   ----------- ------------------


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: c:\Users\CPU12495\.pyenv\pyenv-win\versions\3.12.9\python.exe -m pip install --upgrade pip


In [3]:
import os
os.environ["WANDB_DISABLED"] = "true"
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import argparse
import warnings
warnings.filterwarnings("ignore")


c:\Users\CPU12495\.pyenv\pyenv-win\versions\3.12.9\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# data processing, desmontrator:

In [4]:
df = pd.read_parquet("./task_A/train.parquet")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Dtypes:\n{df.dtypes}")
print(f"\nNull counts:\n{df.isnull().sum()}")
print(f"\nLabel mapping: 0=human, 1=machine")
print(f"\nSample:\n{df.head(10)}")

Shape: (500000, 4)
Columns: ['code', 'generator', 'label', 'language']
Dtypes:
code           str
generator      str
label        int64
language       str
dtype: object

Null counts:
code         0
generator    0
label        0
language     0
dtype: int64

Label mapping: 0=human, 1=machine

Sample:
                                                code  \
0  (a, b, c, d) = [int(x) for x in input().split(...   
1  valid version for the language; all others can...   
2  python\ndef min_cards_to_flip(s):\n    vowels ...   
3  T = int(input())\nfor t in range(T):\n\tcolor ...   
4  for i in range(int(input())):\n\tinput()\n\ta ...   
5  n = int(input())\na = [int(x) for x in input()...   
6  l=[]\nfor i in range(int(input())):\n    n = i...   
7  def is_wilson_prime(p):\n    if not isinstance...   
8  def can_divide_watermelon(w):\n    # If the wa...   
9  def math(s):\n\tif len(s) <= 100:\n\t\tfor i i...   

                             generator  label language  
0                         

We suggest using a single class, it will make refinement easier. 

In your implementation, feel free to update the training procedure, change model and do whatever feels right 

In [ ]:
class CodeBERTTrainer:    
    def __init__(self, max_length=512, model_name="microsoft/codebert-base"):       
         self.max_length = max_length   
              self.model_name = model_name        
                       self.tokenizer = None 
                              self.model = None       
                               self.num_labels = None         
                                  def load_and_prepare_data(self):            
                                        try:       
                                                df = pd.read_parquet("./task_A/train.parquet")           
                                                print(f"Dataset columns: {df.columns.tolist()}")         
                                                 print(f"Sample data:\n{df.head()}")                    
                                                if 'code' not in df.columns or 'label' not in df.columns:       
                                                                                 raise ValueError("Dataset must contain 'code' and 'label' columns")   
                                                                    df = df.dropna(subset=['code', 'label'])               
                                                                    df['label'] = df['label'].astype(int)            self.num_labels = df['label'].nunique()                        print(f"Number of unique labels: {self.num_labels}")            print(f"Label range: {df['label'].min()} to {df['label'].max()}")            print(f"Label distribution:\n{df['label'].value_counts().sort_index()}")            val_df = pd.read_parquet('./task_A/validation.parquet')                        print(f"Train samples: {len(df)}, Validation samples: {len(val_df)}")                        return df, val_df                    except Exception as e:            print(f"Error loading dataset: {e}")            raise        def initialize_model_and_tokenizer(self):        print(f"Initializing {self.model_name} model and tokenizer...")                self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)                self.model = RobertaForSequenceClassification.from_pretrained(            self.model_name,            num_labels=self.num_labels,            problem_type="single_label_classification"        ).to("cuda" if torch.cuda.is_available() else "cpu")                print(f"Model initialized with {self.num_labels} labels")        def tokenize_function(self, examples):        return self.tokenizer(            examples['code'],            truncation=True,            padding=True,            max_length=self.max_length,            return_tensors="pt"        )        def prepare_datasets(self, train_df, val_df):        print("Preparing datasets for training...")                # Reset index to avoid __index_level_0__ column        train_df = train_df.reset_index(drop=True)        val_df = val_df.reset_index(drop=True)                train_dataset = Dataset.from_pandas(train_df[['code', 'label']])        val_dataset = Dataset.from_pandas(val_df[['code', 'label']])                # Remove any index columns if they exist        cols_to_remove = ['code']        if '__index_level_0__' in train_dataset.column_names:            cols_to_remove.append('__index_level_0__')                train_dataset = train_dataset.map(            self.tokenize_function,            batched=True,            remove_columns=cols_to_remove        )        val_dataset = val_dataset.map(            self.tokenize_function,            batched=True,            remove_columns=cols_to_remove        )                train_dataset = train_dataset.rename_column('label', 'labels')        val_dataset = val_dataset.rename_column('label', 'labels')                return train_dataset, val_dataset        def compute_metrics(self, eval_pred):        predictions, labels = eval_pred        predictions = np.argmax(predictions, axis=1)                accuracy = accuracy_score(labels, predictions)        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')                return {            'accuracy': accuracy,            'f1': f1,            'precision': precision,            'recall': recall        }        def train(self, train_dataset, val_dataset, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):        print("Starting training...")        print(self.model)        print(self.model.device)        training_args = TrainingArguments(            output_dir=output_dir,            num_train_epochs=num_epochs,            per_device_train_batch_size=batch_size,            per_device_eval_batch_size=batch_size,            # warmup_steps=500,            weight_decay=0.01,            logging_dir='./logs',            logging_steps=5,            evaluation_strategy="steps",            eval_steps=500,            save_strategy="steps",            save_steps=500,            load_best_model_at_end=True,            metric_for_best_model="f1",            greater_is_better=True,            remove_unused_columns=False,            learning_rate=learning_rate,            lr_scheduler_type="linear",            save_total_limit=2,            report_to=[],        )                data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)                trainer = Trainer(            model=self.model,            args=training_args,            train_dataset=train_dataset,            eval_dataset=val_dataset,            tokenizer=self.tokenizer,            data_collator=data_collator,            compute_metrics=self.compute_metrics,            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]        )        print(f"Start training")        trainer.train()                trainer.save_model()        self.tokenizer.save_pretrained(output_dir)                print(f"Training completed. Model saved to {output_dir}")                return trainer        def evaluate_model(self, trainer, val_dataset):        print("Evaluating model...")                predictions = trainer.predict(val_dataset)        y_pred = np.argmax(predictions.predictions, axis=1)        y_true = predictions.label_ids                print("Classification Report:")        print(classification_report(y_true, y_pred))                return predictions        def run_full_pipeline(self, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):        try:            train_df, val_df = self.load_and_prepare_data()                        self.initialize_model_and_tokenizer()                        train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)                        trainer = self.train(                train_dataset, val_dataset,                 output_dir=output_dir,                num_epochs=num_epochs,                batch_size=batch_size,                learning_rate=learning_rate            )                        self.evaluate_model(trainer, val_dataset)                        print("Pipeline completed successfully!")            return trainer                    except Exception as e:            print(f"Error in pipeline: {e}")            raise    

In [6]:
OUTPUT_DIR = "taskA-model"

trainer_obj = CodeBERTTrainer(
    max_length=256, 
)

t = trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=10,
    batch_size=16,
    learning_rate=2e-5
)


Dataset columns: ['code', 'generator', 'label', 'language']
Sample data:
                                                code  \
0  (a, b, c, d) = [int(x) for x in input().split(...   
1  valid version for the language; all others can...   
2  python\ndef min_cards_to_flip(s):\n    vowels ...   
3  T = int(input())\nfor t in range(T):\n\tcolor ...   
4  for i in range(int(input())):\n\tinput()\n\ta ...   

                        generator  label language  
0                           human      0   Python  
1         Qwen/Qwen2.5-Coder-1.5B      1   Python  
2  Qwen/Qwen2.5-Coder-7B-Instruct      1   Python  
3                           human      0   Python  
4                           human      0   Python  
Number of unique labels: 2
Label range: 0 to 1
Label distribution:
label
0    238475
1    261525
Name: count, dtype: int64
Train samples: 500000, Validation samples: 100000
Initializing microsoft/codebert-base model and tokenizer...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Error in pipeline: Torch not compiled with CUDA enabled


AssertionError: Torch not compiled with CUDA enabled

In [25]:
import torch
import logging
from itertools import chain
from datasets import load_dataset
from tqdm import tqdm


@torch.no_grad()
def predict_with_trainer(trainer_obj, parquet_path, output_path, max_length=512, batch_size=16, device=None):
    """
    Uses trainer_obj.model and trainer_obj.tokenizer to run streaming inference
    over a parquet file with columns ['ID','code'] and writes 'ID,prediction' CSV.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Pull model & tokenizer from your trainer object
    model = trainer_obj.model
    tokenizer = trainer_obj.tokenizer if hasattr(trainer_obj, "tokenizer") else trainer_obj.args._setup_devices and None
    if tokenizer is None and hasattr(trainer_obj, "tokenizer"):
        tokenizer = trainer_obj.tokenizer
    if tokenizer is None:
        raise ValueError("trainer_obj must have a tokenizer (e.g., provided when creating the Trainer).")

    model.to(device)
    model.eval()

    # Stream parquet (no RAM blowup)
    ds = load_dataset("parquet", data_files=parquet_path, split="train", streaming=True)

    # Validate schema and re-chain the first row back into the stream
    it = iter(ds)
    first = next(it)
    if not {"ID", "code"}.issubset(first.keys()):
        raise ValueError("Parquet file must contain 'ID' and 'code' columns")
    stream = chain([first], it)

    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    with open(output_path, "w") as f:
        f.write("ID,prediction\n")

        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            codes = [row["code"] for row in batch]
            ids   = [row["ID"] for row in batch]

            enc = tokenizer(
                codes,
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred_labels = logits.argmax(dim=-1).cpu().tolist()

            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    print(f"Predictions saved to {output_path}")


In [26]:
# After training:
# trainer_obj = CodeBERTTrainer(...).run_full_pipeline(output_dir=..., ...)

TEST_PARQUET = "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A/test.parquet"  # adjust if needed
OUT_CSV = "/kaggle/working/submission.csv"

predict_with_trainer(
    trainer_obj=t,          
    parquet_path=TEST_PARQUET,
    output_path=OUT_CSV,
    max_length=256,
    batch_size=32,
    device="cuda"              
)

print("Wrote:", OUT_CSV)


Predicting: 32it [00:09,  3.35it/s]

Predictions saved to /kaggle/working/submission.csv
Wrote: /kaggle/working/submission.csv


## Option 1: Quick Test Run (1000 samples, 1 epoch)

In [ ]:
# Quick test: 1000 samples, 1 epoch with FULL LOGGING
OUTPUT_DIR_TEST = "taskA-model-test"

print("="*60)
print("🚀 STARTING QUICK TEST RUN")
print("="*60)
print(f"📊 Configuration:")
print(f"   - Samples: 1000 (train) + 200 (val)")
print(f"   - Epochs: 1")
print(f"   - Batch size: 16")
print(f"   - Max length: 256 tokens")
print(f"   - Learning rate: 2e-5")
print(f"   - Output dir: {OUTPUT_DIR_TEST}")
print("="*60)

trainer_test = CodeBERTTrainerTest(max_length=256)

import time
start_time = time.time()

try:
    t_test = trainer_test.run_full_pipeline(
        output_dir=OUTPUT_DIR_TEST,
        num_epochs=1,
        batch_size=16,
        learning_rate=2e-5,
        sample_size=100
    )
    
    elapsed = time.time() - start_time
    print("\n" + "="*60)
    print("✅ TEST RUN COMPLETED SUCCESSFULLY!")
    print("="*60)
    print(f"⏱️  Total time: {elapsed/60:.2f} minutes ({elapsed:.0f} seconds)")
    print(f"📁 Model saved to: {OUTPUT_DIR_TEST}")
    print("="*60)
    print("\n💡 Next steps:")
    print("   1. Check the metrics above (F1, accuracy)")
    print("   2. If results look good, run Option 3 (Medium) or Option 2 (Full)")
    print("   3. If you see errors, check GPU memory or reduce batch_size")
    print("="*60)
    
except Exception as e:
    elapsed = time.time() - start_time
    print("\n" + "="*60)
    print("❌ TEST RUN FAILED!")
    print("="*60)
    print(f"⏱️  Time before error: {elapsed/60:.2f} minutes")
    print(f"🔥 Error: {str(e)}")
    print("="*60)
    print("\n💡 Common fixes:")
    print("   - CUDA out of memory → Reduce batch_size to 8 or 4")
    print("   - Module not found → pip install missing package")
    print("   - Dataset error → Check file paths")
    print("="*60)
    raise

## Option 2: Full Training (All data, 10 epochs) — Run this after test succeeds

In [ ]:
# Full training - uncomment when ready
OUTPUT_DIR = "taskA-model"

trainer_obj = CodeBERTTrainer(
    max_length=256, 
)

t = trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=10,      # Full 10 epochs
    batch_size=16,
    learning_rate=2e-5
)

print("\n✅ Full training completed!")

## Option 3: Medium Run (10K samples, 3 epochs) — Balanced option

In [ ]:
# Medium run: 10K samples, 3 epochs
OUTPUT_DIR_MEDIUM = "taskA-model-medium"

trainer_medium = CodeBERTTrainerTest(max_length=256)

t_medium = trainer_medium.run_full_pipeline(
    output_dir=OUTPUT_DIR_MEDIUM,
    num_epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    sample_size=10000
)

print("\n✅ Medium run completed!")